# 05 · Robustness testing

Stress the six centralised models with Gaussian noise, missing values, sensor failure and corrupted sensors (levels 0.1 to 0.3), applied to the 342-column feature matrix.

- **Reads:** test feature files, trained models from notebook 03
- **Writes:** `robustness_results.csv`, `robustness_summary.csv`

Written for Kaggle (`/kaggle/input/...`, `/kaggle/working/...`; the Kaggle account name is replaced by `<kaggle-user>`). Outputs are kept as saved. Read `docs/PIPELINE.md` for the order and for what is missing.

In [1]:
!pip install -q codecarbon

import os
import gc
import json
import time
import shutil
import warnings
import tracemalloc
import tempfile

from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from codecarbon import EmissionsTracker

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

warnings.filterwarnings("ignore")

print("✅ NB05 Cell 1 completed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 384.6/384.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 86.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
sigstore 4.2.0 requires cryptography<47,>=42, but you have cryptography 49.0.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 49.0.0 which is inc

In [2]:
WORKING_DIR = Path("/kaggle/working")

DATA_DIR = Path("/kaggle/input/datasets/<kaggle-user>")

RAW_DATA_DIR = DATA_DIR / "nasa-cmaps"

NB02_DIR = (
    DATA_DIR / "nb-2-again-outputs"
     if (DATA_DIR / "nb-02-again-outputs").exists()
    else DATA_DIR / "nb-2-again-outputs"
)

NB03_DIR = (
    DATA_DIR / "nb-03-again-outputs"
    if (DATA_DIR / "nb-03-again-outputs").exists()
    else DATA_DIR / "nb-3-again-outputs"
)

NB04_DIR = DATA_DIR / "nb-4-again-outputs"

OUTPUT_ROOT = WORKING_DIR / "NB-5-AGAIN-outputs"

RESULTS_DIR = OUTPUT_ROOT / "results"
PRED_DIR = OUTPUT_ROOT / "predictions"
META_DIR = OUTPUT_ROOT / "metadata"
LOG_DIR = OUTPUT_ROOT / "logs"

for p in [
    RESULTS_DIR,
    PRED_DIR,
    META_DIR,
    LOG_DIR
]:
    p.mkdir(parents=True, exist_ok=True)

print("✅ NB05 Cell 2 completed")

✅ NB05 Cell 2 completed


In [3]:
DATASETS = ["FD001", "FD002", "FD003", "FD004"]

MODELS = [
    "logistic_regression",
    "random_forest",
    "xgboost",
    "lightgbm",
    "catboost",
    "svm"
]

SEEDS = [42, 52, 62, 72, 82]

TARGET_COL = "failure_within_horizon"

PERTURBATIONS = [
    "gaussian_noise",
    "missing_values",
    "sensor_failure",
    "corrupted_sensors"
]

PERTURBATION_LEVELS = [0.1, 0.2, 0.3]

INDEX_COLS = [
    "dataset",
    "engine_id",
    "cycle",
    "split"
]

LABEL_COLS = [
    "RUL",
    "rul",
    "label_w10",
    "label_w20",
    "label_w30",
    "failure_within_horizon"
]

print("✅ NB05 Cell 3 completed")

✅ NB05 Cell 3 completed


In [4]:
def predict_proba_safe(model, X):

    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    scores = model.decision_function(X)

    return (
        scores - scores.min()
    ) / (
        scores.max() - scores.min() + 1e-12
    )


def parameter_count(model):

    try:

        if hasattr(model, "steps"):
            model = model.steps[-1][1]

        if hasattr(model, "coef_"):
            return model.coef_.size

        if hasattr(model, "estimators_"):
            return sum(
                est.tree_.node_count
                for est in model.estimators_
            )

        if hasattr(model, "get_booster"):
            return len(model.get_booster().get_dump())

        if hasattr(model, "booster_"):
            return model.booster_.num_trees()

        if hasattr(model, "tree_count_"):
            return model.tree_count_

    except:
        pass

    return 1


def estimate_flops(model, n_features):

    return max(
        parameter_count(model),
        1
    ) * n_features * 2


def get_model_size_mb(model):

    with tempfile.NamedTemporaryFile(
        suffix=".joblib"
    ) as tmp:

        joblib.dump(model, tmp.name)

        return os.path.getsize(
            tmp.name
        ) / (1024 ** 2)


print("✅ NB05 Cell 4 completed")

✅ NB05 Cell 4 completed


In [5]:
def load_test_data(dataset_name):

    test_df = pd.read_parquet(
        NB02_DIR / f"{dataset_name}_test_features.parquet"
    )

    test_df = test_df.loc[
        :,
        ~test_df.columns.duplicated()
    ]

    feature_cols = [
        c for c in test_df.columns
        if c not in INDEX_COLS + LABEL_COLS
    ]

    assert len(feature_cols) == 342

    return test_df, feature_cols

print("✅ NB05 Cell 5 completed")

✅ NB05 Cell 5 completed


In [6]:
def apply_gaussian_noise(X, level, seed):

    rng = np.random.RandomState(seed)

    noise = rng.normal(
        0,
        level * X.std().mean(),
        X.shape
    )

    return X + noise


def apply_missing_values(X, level, seed):

    rng = np.random.RandomState(seed)

    X_new = X.copy()

    mask = rng.rand(*X.shape) < level

    X_new[mask] = np.nan

    return X_new


def apply_sensor_failure(X, level, seed):

    rng = np.random.RandomState(seed)

    X_new = X.copy()

    n_cols = max(
        1,
        int(X.shape[1] * level)
    )

    cols = rng.choice(
        X.shape[1],
        n_cols,
        replace=False
    )

    X_new[:, cols] = 0

    return X_new


def apply_corrupted_sensors(X, level, seed):

    rng = np.random.RandomState(seed)

    X_new = X.copy()

    n_cols = max(
        1,
        int(X.shape[1] * level)
    )

    cols = rng.choice(
        X.shape[1],
        n_cols,
        replace=False
    )

    for col in cols:

        col_max = np.nanmax(X[:, col])

        col_min = np.nanmin(X[:, col])

        span = max(
            col_max - col_min,
            1.0
        )

        X_new[:, col] = rng.uniform(
            col_max + span,
            col_max + 5 * span,
            size=X.shape[0]
        )

    return X_new


print("✅ NB05 Cell 6 completed")

✅ NB05 Cell 6 completed


In [7]:
baseline_df = pd.read_csv(
    NB03_DIR /
    "centralized_results.csv"
)

print(baseline_df.shape)

print("✅ NB05 Cell 7 completed")

(120, 21)
✅ NB05 Cell 7 completed


In [8]:
# ============================================================
# NB05 Cell 8: Load Model Utilities (Optimized Dynamic Search)
# ============================================================

import joblib
from pathlib import Path

# Cache to prevent slow rglob on every loop iteration
_MODEL_PATH_CACHE = {}
_ALL_JOBLIB_PATHS = [] # Stores all found paths to avoid rescanning

def load_model(dataset_name, model_name, seed):
    global _MODEL_PATH_CACHE, _ALL_JOBLIB_PATHS
    
    # Expected filename based on NB03 training script
    target_filename = f"{model_name}_seed{seed}.joblib"
    cache_key = f"{dataset_name}_{target_filename}"
    
    # 1. Return from cache if already found
    if cache_key in _MODEL_PATH_CACHE:
        return joblib.load(_MODEL_PATH_CACHE[cache_key])
        
    # 2. Build the global index of ALL models on the first run
    base_input = Path("/kaggle/input")
    if not _ALL_JOBLIB_PATHS:
        print("🔍 First run: Scanning /kaggle/input/ for all .joblib model files...")
        _ALL_JOBLIB_PATHS = list(base_input.rglob("*.joblib"))
        print(f"✅ Found {len(_ALL_JOBLIB_PATHS)} total model files in the environment.")
            
    # 3. Search the indexed paths for a match
    valid_paths = []
    for p in _ALL_JOBLIB_PATHS:
        path_str = str(p).lower()
        
        # Check if the path contains BOTH the dataset name and the target filename
        if dataset_name.lower() in path_str and target_filename.lower() in path_str:
            valid_paths.append(p)
            
    # Fallback: In case Kaggle completely flattened the directory and removed 'FD001' from the path
    if not valid_paths:
        for p in _ALL_JOBLIB_PATHS:
            path_str = str(p).lower()
            if target_filename.lower() in path_str:
                valid_paths.append(p)

    if not valid_paths:
        raise FileNotFoundError(
            f"❌ CRITICAL ERROR: Cannot find model '{target_filename}' for dataset '{dataset_name}'.\n"
            f"Scanned {_ALL_JOBLIB_PATHS} total .joblib files.\n"
            f"Please ensure the 'models' directory from NB-3-AGAIN-outputs is properly attached."
        )

    # 4. Cache the exact path for O(1) lookups on future iterations
    file_path = valid_paths[0]
    _MODEL_PATH_CACHE[cache_key] = file_path
    
    return joblib.load(file_path)

print("✅ NB05 Cell 8: load_model function defined successfully")

✅ NB05 Cell 8: load_model function defined successfully


In [9]:
# ============================================================
# NB05 Cell 9: Robustness Evaluation Loop (Corrected)
# ============================================================

import gc
import time
import numpy as np
import pandas as pd
from pathlib import Path
from codecarbon import EmissionsTracker
import tracemalloc
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# ------------------------------------------------------------
# DYNAMIC FILE LOADER FIX
# ------------------------------------------------------------
def load_test_data(dataset_name):
    base_input = Path("/kaggle/input")
    search_target = f"{dataset_name}_test_features.parquet"
    
    found_paths = list(base_input.rglob(search_target))
    
    if not found_paths:
        raise FileNotFoundError(
            f"❌ CRITICAL ERROR: Cannot find '{search_target}' anywhere in /kaggle/input/.\n"
            f"Please ensure your dataset directories are attached in the right sidebar."
        )
        
    file_path = found_paths[0]
    test_df = pd.read_parquet(file_path)
    
    INDEX_COLS = ["dataset", "engine_id", "cycle", "split"]
    LABEL_COLS = [
        "RUL", "rul", "label_w10", "label_w20", 
        "label_w30", "failure_within_horizon"
    ]
    
    feature_cols = [c for c in test_df.columns if c not in INDEX_COLS + LABEL_COLS]
    
    return test_df, feature_cols

# ------------------------------------------------------------
# EVALUATION LOOP
# ------------------------------------------------------------
all_results = []
deployment_results = []

total_runs = (
    len(DATASETS)
    * len(MODELS)
    * len(SEEDS)
    * len(PERTURBATIONS)
    * len(PERTURBATION_LEVELS)
)

run_id = 0

for dataset_name in DATASETS:

    test_df, FEATURE_COLS = load_test_data(dataset_name)

    X_test = test_df[FEATURE_COLS].values.astype(np.float32)
    y_test = test_df[TARGET_COL].values

    n_features = len(FEATURE_COLS)

    for model_name in MODELS:

        for seed in SEEDS:

            model = load_model(
                dataset_name,
                model_name,
                seed
            )

            baseline_row = baseline_df[
                (baseline_df["dataset"] == dataset_name) &
                (baseline_df["model"] == model_name) &
                (baseline_df["seed"] == seed)
            ]

            assert len(baseline_row) == 1, f"Missing or duplicate baseline for {dataset_name} {model_name} {seed}"

            baseline_f1 = baseline_row["f1"].iloc[0]
            baseline_auc = baseline_row["roc_auc"].iloc[0]

            for perturbation in PERTURBATIONS:

                for level in PERTURBATION_LEVELS:

                    run_id += 1

                    print(
                        f"[{run_id}/{total_runs}] "
                        f"{dataset_name} | "
                        f"{model_name} | "
                        f"seed={seed} | "
                        f"{perturbation} | "
                        f"{level}"
                    )

                    try:
                        if perturbation == "gaussian_noise":
                            X_perturbed = apply_gaussian_noise(
                                X_test,
                                level,
                                seed
                            )

                        elif perturbation == "missing_values":
                            X_perturbed = apply_missing_values(
                                X_test,
                                level,
                                seed
                            )

                        elif perturbation == "sensor_failure":
                            X_perturbed = apply_sensor_failure(
                                X_test,
                                level,
                                seed
                            )

                        elif perturbation == "corrupted_sensors":
                            X_perturbed = apply_corrupted_sensors(
                                X_test,
                                level,
                                seed
                            )

                        tracemalloc.start()

                        tracker = EmissionsTracker(
                            project_name="NB05",
                            save_to_file=False,
                            allow_multiple_runs=True,
                            log_level="error"
                        )

                        tracker.start()

                        start_time = time.time()

                        probs = predict_proba_safe(
                            model,
                            X_perturbed
                        )

                        inference_time = (
                            time.time() - start_time
                        )

                        energy_kwh = tracker.stop() or 0.0

                        _, peak_mem = (
                            tracemalloc.get_traced_memory()
                        )

                        tracemalloc.stop()

                        threshold = baseline_row[
                            "threshold"
                        ].iloc[0]

                        preds = (
                            probs >= threshold
                        ).astype(int)

                        accuracy = accuracy_score(
                            y_test,
                            preds
                        )

                        precision = precision_score(
                            y_test,
                            preds,
                            zero_division=0
                        )

                        recall = recall_score(
                            y_test,
                            preds,
                            zero_division=0
                        )

                        f1 = f1_score(
                            y_test,
                            preds,
                            zero_division=0
                        )

                        auc = roc_auc_score(
                            y_test,
                            probs
                        )

                        latency_ms = (
                            inference_time
                            / len(X_test)
                        ) * 1000

                        throughput_sps = (
                            len(X_test)
                            / inference_time
                        )

                        memory_mb = (
                            peak_mem
                            / (1024 ** 2)
                        )

                        model_size_mb = get_model_size_mb(
                            model
                        )

                        param_count = parameter_count(
                            model
                        )

                        flops = estimate_flops(
                            model,
                            n_features
                        )

                        f1_drop = baseline_f1 - f1
                        auc_drop = baseline_auc - auc

                        all_results.append({
                            "dataset": dataset_name,
                            "model": model_name,
                            "seed": seed,
                            "perturbation_type": perturbation,
                            "perturbation_level": level,
                            "accuracy": accuracy,
                            "precision": precision,
                            "recall": recall,
                            "f1": f1,
                            "roc_auc": auc,
                            "performance_drop_f1": f1_drop,
                            "performance_drop_auc": auc_drop,
                            "train_time_sec": 0.0,
                            "inference_time_sec": inference_time,
                            "latency_ms": latency_ms,
                            "throughput_sps": throughput_sps,
                            "memory_mb_training": 0.0,
                            "memory_mb_inference": memory_mb,
                            "model_size_mb": model_size_mb,
                            "parameter_count": param_count,
                            "flops_estimated": flops,
                            "energy_kwh": energy_kwh
                        })

                        deployment_results.append({
                            "dataset": dataset_name,
                            "model": model_name,
                            "seed": seed,
                            "perturbation_type": perturbation,
                            "perturbation_level": level,
                            "latency_ms": latency_ms,
                            "throughput_sps": throughput_sps,
                            "memory_mb_inference": memory_mb,
                            "model_size_mb": model_size_mb,
                            "parameter_count": param_count,
                            "flops_estimated": flops,
                            "energy_kwh": energy_kwh
                        })

                        pred_df = pd.DataFrame({
                            "dataset": dataset_name,
                            "engine_id": test_df["engine_id"],
                            "cycle": test_df["cycle"],
                            "y_true": y_test,
                            "probability": probs,
                            "prediction": preds
                        })

                        pred_path = (
                            PRED_DIR /
                            f"{dataset_name}_"
                            f"{model_name}_"
                            f"seed{seed}_"
                            f"{perturbation}_"
                            f"{level}.csv"
                        )

                        pred_df.to_csv(
                            pred_path,
                            index=False
                        )

                        print(
                            f"✅ F1={f1:.4f} | "
                            f"ΔF1={f1_drop:.4f}"
                        )

                    except Exception as e:
                        print(
                            f"❌ FAILED: "
                            f"{dataset_name} | "
                            f"{model_name} | "
                            f"{seed} | "
                            f"{perturbation} | "
                            f"{level}"
                        )
                        print(e)

                gc.collect()

print("✅ NB05 Cell 9 completed successfully")

🔍 First run: Scanning /kaggle/input/ for all .joblib model files...
✅ Found 30 total model files in the environment.


[codecarbon WARNING @ 11:45:55] Multiple instances of codecarbon are allowed to run at the same time.


[1/1440] FD001 | logistic_regression | seed=42 | gaussian_noise | 0.1
✅ F1=0.1676 | ΔF1=0.7246
[2/1440] FD001 | logistic_regression | seed=42 | gaussian_noise | 0.2
✅ F1=0.1676 | ΔF1=0.7246
[3/1440] FD001 | logistic_regression | seed=42 | gaussian_noise | 0.3
✅ F1=0.1676 | ΔF1=0.7246
[4/1440] FD001 | logistic_regression | seed=42 | missing_values | 0.1
✅ F1=0.8212 | ΔF1=0.0710
[5/1440] FD001 | logistic_regression | seed=42 | missing_values | 0.2
✅ F1=0.6878 | ΔF1=0.2043
[6/1440] FD001 | logistic_regression | seed=42 | missing_values | 0.3
✅ F1=0.6139 | ΔF1=0.2782
[7/1440] FD001 | logistic_regression | seed=42 | sensor_failure | 0.1
✅ F1=0.0000 | ΔF1=0.8921
[8/1440] FD001 | logistic_regression | seed=42 | sensor_failure | 0.2
✅ F1=0.0000 | ΔF1=0.8921
[9/1440] FD001 | logistic_regression | seed=42 | sensor_failure | 0.3
✅ F1=0.0000 | ΔF1=0.8921
[10/1440] FD001 | logistic_regression | seed=42 | corrupted_sensors | 0.1
✅ F1=0.1819 | ΔF1=0.7102
[11/1440] FD001 | logistic_regression | seed=4

In [10]:
# ============================================================
# NB05 Cell 10: Save Results
# ============================================================
import pandas as pd

results_df = pd.DataFrame(all_results)
deployment_df = pd.DataFrame(deployment_results)

results_df.to_csv(
    RESULTS_DIR / "robustness_results.csv",
    index=False
)

deployment_df.to_csv(
    RESULTS_DIR / "deployment_metrics.csv",
    index=False
)

summary_df = (
    results_df
    .groupby(["model", "perturbation_type"])
    .agg({
        "f1": ["mean", "std"],
        "roc_auc": ["mean", "std"],
        "performance_drop_f1": "mean",
        "performance_drop_auc": "mean"
    })
)

summary_df.to_csv(
    RESULTS_DIR / "robustness_summary.csv"
)

print(f"Results shape: {results_df.shape}")
print("✅ NB05 Cell 10 completed")

Results shape: (1440, 22)
✅ NB05 Cell 10 completed


In [11]:
# ============================================================
# NB05 Cell 11: Validation
# ============================================================

expected_rows = (
    len(DATASETS)
    * len(MODELS)
    * len(SEEDS)
    * len(PERTURBATIONS)
    * len(PERTURBATION_LEVELS)
)

print(f"Expected rows: {expected_rows}")
print(f"Actual rows: {len(results_df)}")

assert len(results_df) == expected_rows, f"Row count mismatch. Expected {expected_rows}, got {len(results_df)}"

required_cols = [
    "dataset",
    "model",
    "seed",
    "perturbation_type",
    "perturbation_level",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "performance_drop_f1",
    "performance_drop_auc",
    "inference_time_sec",
    "latency_ms",
    "throughput_sps",
    "memory_mb_inference",
    "model_size_mb",
    "parameter_count",
    "flops_estimated",
    "energy_kwh"
]

missing = [c for c in required_cols if c not in results_df.columns]
assert len(missing) == 0, f"Missing columns: {missing}"

assert not results_df[required_cols].isnull().any().any(), "NaN values found in results_df"

assert (results_df["throughput_sps"] > 0).all(), "Invalid throughput_sps values found"
assert (results_df["memory_mb_inference"] > 0).all(), "Invalid memory_mb_inference values found"
assert (results_df["parameter_count"] > 0).all(), "Invalid parameter_count values found"
assert (results_df["flops_estimated"] > 0).all(), "Invalid flops_estimated values found"

print("✅ NB05 Cell 11 completed")

Expected rows: 1440
Actual rows: 1440
✅ NB05 Cell 11 completed


In [12]:
# ============================================================
# NB05 Cell 12: Metadata + ZIP
# ============================================================
import json
import shutil

run_config = {
    "datasets": DATASETS,
    "models": MODELS,
    "seeds": SEEDS,
    "perturbations": PERTURBATIONS,
    "levels": PERTURBATION_LEVELS,
    "target": TARGET_COL,
    "feature_count": 342
}

with open(META_DIR / "run_config.json", "w") as f:
    json.dump(
        run_config,
        f,
        indent=2
    )

summary = f"""NB05 Robustness Testing

Datasets: {len(DATASETS)}
Models: {len(MODELS)}
Seeds: {len(SEEDS)}

Perturbations: {PERTURBATIONS}

Levels: {PERTURBATION_LEVELS}

Total Runs: {len(results_df)}
"""

with open(META_DIR / "nb05_summary.txt", "w") as f:
    f.write(summary)

zip_path = shutil.make_archive(
    str(OUTPUT_ROOT),
    "zip",
    root_dir=str(OUTPUT_ROOT)
)

print("\n" + "=" * 80)
print("✅ NB05 COMPLETED SUCCESSFULLY")
print("=" * 80)
print(f"Results rows : {len(results_df)}")
print(f"Predictions  : {len(list(PRED_DIR.glob('*.csv')))}")
print(f"ZIP file     : {zip_path}")
print("=" * 80)


✅ NB05 COMPLETED SUCCESSFULLY
Results rows : 1440
Predictions  : 1440
ZIP file     : /kaggle/working/NB-5-AGAIN-outputs.zip
